**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# APIs in Python: World Bank data

In class, we used the World Bank API as the data source for a live Vega-Lite chart. Here we use the **same API** from Python.

Why? In Python we can **check** the data, **clean** it and **save** it. And because it's code, we can re-run the notebook any time to get the latest data.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import requests        # fetch things from the internet
import pandas as pd     # work with tables of data
import altair as alt    # make charts

<br><br>

---

<br>

## 1. The API URL

This is the same URL we used in class: South Korea's fertility rate (births per woman).

| Part | Meaning |
|---|---|
| `https://api.worldbank.org/v2/` | The World Bank API (version 2) |
| `country/kor/` | Which country (a 3-letter code) |
| `indicator/SP.DYN.TFRT.IN` | Which data series (code from [data.worldbank.org](https://data.worldbank.org/indicator/SP.DYN.TFRT.IN)) |
| `?format=json&per_page=100` | Options: send JSON, and up to 100 rows (the default is only 50) |

> **Tip:** paste the URL into your browser first. If you can see data there, Python can get it too.

In [ ]:
url = "https://api.worldbank.org/v2/country/kor/indicator/SP.DYN.TFRT.IN?format=json&per_page=100"

<br><br>

---

<br>

## 2. Fetch the data

`requests.get()` sends the request. `<Response [200]>` means it worked.

In [ ]:
response = requests.get(url)
response

The World Bank sends JSON. `.json()` turns it into Python lists and dictionaries:

In [ ]:
data = response.json()

<br>

The response is a **list with two items**:
- `data[0]`: information **about** the data (the *metadata*)
- `data[1]`: the data itself

This is why our Vega-Lite chart needed `"format": {"property": "[1]"}`: it tells Vega-Lite to use the second item.

In [ ]:
data[0]

> **Always check the metadata.** `total` is how many rows exist. If it's bigger than `per_page`, you haven't got all the data (this is why we added `per_page=100`).

In [ ]:
data[1][0]    # the first row of data

Each row is a dictionary. Some values are dictionaries **inside** dictionaries: for example, the country name is inside `country`.

<br><br>

---

<br>

## 3. Turn it into a table

`pd.json_normalize()` turns a list of dictionaries into a table. It also **flattens** the nested parts: the country name becomes a column called `country.value`.

In [ ]:
df = pd.json_normalize(data[1])
df.head()

We only need three columns. Then we tidy up, using the same steps as Week 3:

In [ ]:
df = df[['country.value', 'date', 'value']].copy()

df = df.dropna(subset=['value'])                         # remove years with no data yet
df['date'] = pd.to_datetime(df['date'], format='%Y')     # '2024' -> 2024-01-01

df.head()

<br><br>

---

<br>

## 4. Chart it

In [ ]:
chart = alt.Chart(df).mark_line().encode(
    x=alt.X('date:T').title(None),
    y=alt.Y('value:Q').title('Births per woman')
).properties(
    title='South Korea: fertility rate'
)

chart

<br><br>

---

<br>

## 5. Save it

Re-run this notebook next year, and the CSV will include the latest data, with no extra work.

In [ ]:
df.to_csv('worldbank_fertility_kor.csv', index=False)

<br><br>

---

<br>

## 6. The easy way: a package made for this API

Popular APIs often have a **Python package** that does the work for us. For the World Bank, it's [`wbgapi`](https://pypi.org/project/wbgapi/).

It isn't installed in Colab, so we install it first.

> **Be patient:** `wbgapi` uses a slower part of the World Bank API and fetches some background information first, so the cells below can take a minute or more, especially the first time. Later requests in the same session are faster.

In [ ]:
%pip install wbgapi --quiet

import wbgapi as wb

<br>

**Search for an indicator code**, without leaving Python:

In [ ]:
wb.series.info(q='fertility')

<br>

**Get the data in one line.** Here, three countries, with a row for each year:

In [ ]:
df_wb = wb.data.DataFrame('SP.DYN.TFRT.IN', ['KOR', 'JPN', 'GBR'], index='time', numericTimeKeys=True)

df_wb.tail()

<br>

| By hand (sections 1–3) | With `wbgapi` |
|---|---|
| Build the URL ourselves | Just the indicator and country codes |
| Handle `per_page` and the `[metadata, data]` list | Done for us |
| Flatten nested JSON with `json_normalize` | Returns a table directly |
| Works for **any** API | Only works for the World Bank |

> Packages save time, but it's still worth knowing what they do underneath. When there's no package for your API, `requests` + `json_normalize` always works.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Change the URL in section 1 to another country (e.g. `gbr`, `bra`, `nga`) and re-run the notebook.

**2.** Find another indicator on [data.worldbank.org](https://data.worldbank.org/indicator) (e.g. life expectancy, `SP.DYN.LE00.IN`) and chart it.

**3.** Several countries in one request: use `country/kor;jpn;gbr/`. In your chart, add `color=alt.Color('country.value:N')`.

In [ ]:
# Your code here
